In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Import

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder , StandardScaler , LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score , classification_report

import xgboost as xgb

# EDA

In [ ]:
train_data = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")

## EDA on training Data

In [ ]:
train_data.shape

In [ ]:
train_data.head()

In [ ]:
train_data.info()

In [ ]:
train_data.isnull().sum()

In [ ]:
train_data.describe()

In [ ]:
train_data.duplicated().sum()

In [ ]:
# train_data.corr()

## Univariate Analysis on Train Data

In [ ]:
Cat_Cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type',
            'Water_Source','Mulching_Used', 'Region', 'Irrigation_Need' ] 

fig , axes = plt.subplots(3,3, figsize=(18,15))
axes = axes.flatten()


for i,col in enumerate(Cat_Cols):
    train_data[col].value_counts().plot(kind='pie', autopct='%.2f' , ax = axes[i])
    axes[i].set_title(col)
    axes[i].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
Num_Cols = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C',
           'Humidity', 'Rainfall_mm','Sunlight_Hours', 'Wind_Speed_kmh',
            'Field_Area_hectare','Previous_Irrigation_mm' ]
fig , axes = plt.subplots(11,2, figsize=(20, 64))
axes = axes.flatten()
for i ,col in enumerate(Num_Cols):
    idx = i * 2
    sns.boxplot(train_data[col] , ax=axes[idx])
    axes[idx].set_title(col)
    
    sns.histplot(train_data[col] , ax=axes[idx+1])
    axes[idx].set_title(col)
    
plt.tight_layout()
plt.show()
        
     

## Mutltivariate Analysis

In [ ]:
ct = pd.crosstab(train_data['Crop_Growth_Stage'], train_data['Irrigation_Need'] , normalize='index' )
ct.plot(kind='bar', stacked=True )

In [ ]:
train_data.info()

In [ ]:
sns.barplot(x= train_data['Irrigation_Need'] , y=train_data['Temperature_C'])

In [ ]:
sns.barplot(x=train_data['Irrigation_Need'], y=train_data['Soil_Moisture'])

In [ ]:
sns.barplot(x=train_data['Irrigation_Need'], y=train_data['Rainfall_mm'])

In [ ]:
sns.barplot(x=train_data['Irrigation_Need'], y=train_data['Wind_Speed_kmh'])

In [ ]:
pd.crosstab(train_data['Mulching_Used'],train_data['Irrigation_Need'] ,normalize='index').plot(kind='bar', stacked=True)

In [ ]:
pd.crosstab( train_data['Soil_Type'],train_data['Irrigation_Need'], normalize='index').plot(kind='bar', stacked=True)

In [ ]:
pd.crosstab(train_data['Season'], train_data['Irrigation_Need'], normalize='index').plot(kind='bar', stacked=True)

## EDA on Test Data

In [ ]:
test_data.shape

In [ ]:
test_data.head()

In [ ]:
test_data.info()

In [ ]:
test_data.describe()

In [ ]:
test_data.duplicated().sum()

## Univariate Analysis on Test Data

In [ ]:
Cat_Cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type',
            'Water_Source','Mulching_Used', 'Region'] 

fig , axes = plt.subplots(3,3, figsize=(18, 15))
axes = axes.flatten()

for i , col in enumerate(Cat_Cols):
    test_data[col].value_counts().plot(kind='pie', autopct='%.2f' , ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
Num_Cols = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C',
           'Humidity', 'Rainfall_mm','Sunlight_Hours', 'Wind_Speed_kmh',
            'Field_Area_hectare','Previous_Irrigation_mm' ]
fig , axes = plt.subplots(11,2, figsize=(20, 64))
axes = axes.flatten()
for i ,col in enumerate(Num_Cols):
    idx = i * 2
    sns.boxplot(train_data[col] , ax=axes[idx])
    axes[idx].set_title(col)
    
    sns.kdeplot(train_data[col] , ax=axes[idx+1])
    axes[idx].set_title(col)
    
plt.tight_layout()
plt.show()

# Feature Engineering

In [ ]:
train_data.duplicated().sum()

In [ ]:
test_data.duplicated().sum()

In [ ]:
X = train_data.drop('Irrigation_Need', axis=1)
y = train_data['Irrigation_Need']
X_train , X_val , y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape

In [ ]:
X_val.shape

In [ ]:
y_train.shape

## Encoding Categorical Variable

In [ ]:
Cat_Cols = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type',
            'Water_Source','Mulching_Used', 'Region']
Num_Cols = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C',
           'Humidity', 'Rainfall_mm','Sunlight_Hours', 'Wind_Speed_kmh',
            'Field_Area_hectare','Previous_Irrigation_mm' ]

preprocessor = ColumnTransformer(
    transformers=[
    ("cat", OneHotEncoder(handle_unknown='ignore'), Cat_Cols),
    ("num", StandardScaler(), Num_Cols)
    ]
)

In [ ]:
le = LabelEncoder()

In [ ]:
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

# Model and Pipeline

In [ ]:
pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        objective='multi:softmax',
        num_class=3,
        random_state=42,
        n_jobs=-1,
        eval_metric='mlogloss'
    ))
])

In [ ]:
le.classes_

## Hyperparameter Tuning

In [ ]:
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [4, 6, 8],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.7, 0.8, 0.9],
    'model__colsample_bytree': [0.7, 0.8, 0.9]
}

grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_search.fit(X_train , y_train)

In [ ]:
print("Best Parameters: ", grid_search.best_params_)
print("Best Cross-Validation Score: ", grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
best_model.score(X_val, y_val)

In [ ]:
y_preds = best_model.predict(X_val)

In [ ]:
print(classification_report(y_val , y_preds))

# Prediction on Test Data

In [ ]:
test_data.head()

In [ ]:
y_pred = best_model.predict(test_data)

In [ ]:
y_pred

In [ ]:
y_pred_label = le.inverse_transform(y_pred)

In [ ]:
y_pred_label

In [ ]:
submission = pd.DataFrame({
    "id":test_data['id'],
    "Irrigation_Need": y_pred_label
})

In [ ]:
submission

In [ ]:
submission.to_csv("submission.csv", index=False)